In [33]:
from __future__ import annotations

from pathlib import Path
from typing import Any
import json

import pandas as pd
import yaml

def load_events(results_file: str | Path) -> pd.DataFrame:
    """Load events.jsonl file into a single DataFrame."""
    path = Path(results_file)
    frames = []
    rows = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line]
    if rows:
        frames.append(pd.DataFrame(rows))
    if not frames:
        return pd.DataFrame()
    df = pd.concat(frames, ignore_index=True)
    return df

def load_all_events(results_files: list) -> pd.DataFrame:
    all_dfs = []
    for rf in results_files:
        temp_df = load_events(rf)
        all_dfs.append(temp_df)
    return pd.concat(all_dfs, ignore_index=True)


def _load_analysis_config(config: dict[str, Any] | str | Path | None) -> dict[str, Any] | None:
    if config is None or isinstance(config, dict):
        return config
    path = Path(config)
    text = path.read_text(encoding="utf-8")
    if path.suffix.lower() == ".json":
        return json.loads(text)
    return yaml.safe_load(text)


exp_name = 'left-digit'
results_dir = Path('/home/dw/github/toollab/results/')/exp_name

results_files = list(results_dir.glob('*-events.jsonl'))
config_files = list(results_dir.glob('*-config.json'))
config_file = config_files[0] if config_files else None
config = _load_analysis_config(config_file)
metadata_columns = list(config['metadata'].keys())
print('metadata_columns', metadata_columns)


df_raw = load_all_events(results_files)
df = df_raw.copy().sort_values(['session_name', 'step_index'])
meta_df = df['meta'].dropna().apply(pd.Series)
meta_df = meta_df.add_prefix('meta_')
df = df.join(meta_df)

meta_cols = df.filter(like='meta_').columns
df[meta_cols] = df.groupby('session_name')[meta_cols].ffill()

# print(f"{len(df)} events across {df['session_name'].nunique()} sessions")
# print(df[['option_count', 'design']])
print(df.columns)
print(df.shape)

print("\nNumber of sessions per condition:")

df.groupby(['model_name', *metadata_columns])['session_name'].nunique().reset_index()

gemini_flash_lite = df[(df['model_name']=='gemini-3.1-flash-lite-preview')&(df['kind']=='tool')]
# gemini_flash_lite[['budget_type', 'budget_max','choice','step_index','option_id', 'attribute_id', 'value', 'transition']]


metadata_columns ['option_count']
Index(['session_name', 'experiment_name', 'provider', 'model_name',
       'forced_choice', 'choice', 'cumulative_cost_usd',
       'budget_remaining_usd', 'cumulative_cost_tools',
       'budget_remaining_tools', 'option_count', 'seed', 'started_at',
       'finished_at', 'step_index', 'kind', 'content', 'reasoning',
       'input_tokens', 'output_tokens', 'input_cost', 'output_cost',
       'tool_cost', 'tool_calls', 'finish_reason', 'tool_name', 'tool_call_id',
       'option_id', 'attribute_id', 'value', 'turn_cost_tools', 'is_revisit',
       'transition', 'option_label', 'budget_type', 'budget_max',
       'budget_remaining_tokens', 'cumulative_cost_tokens', 'turn_cost_tokens',
       'meta', 'meta_gpt-5.4-mini-2026-03-17', 'meta_none',
       'meta_gpt-5.4-2026-03-05', 'meta_gpt-5.4-nano-2026-03-17',
       'meta_claude-sonnet-4-6', 'meta_thinking', 'meta_effort',
       'meta_model_version'],
      dtype='str')
(230, 48)

Number of sessions per

In [21]:
gemini_flash = df[(df['model_name']=='gemini-3-flash-preview')&(df['kind']=='tool')]
# gemini_lite.columns
gemini_flash[['budget_type', 'budget_max','choice','step_index','option_id', 'attribute_id', 'value', 'transition']]


,budget_type,budget_max,choice,step_index,option_id,attribute_id,value,transition
1,NaN,NaN,coffee_a,2,coffee_a,price_dollars,$12,first
3,NaN,NaN,coffee_a,4,coffee_b,price_dollars,$13,attribute
5,NaN,NaN,coffee_a,6,coffee_a,weight_oz,13.0 oz,diagonal
7,NaN,NaN,coffee_a,8,coffee_b,weight_oz,13.9 oz,attribute
9,NaN,NaN,coffee_a,10,coffee_a,price_cents,.99,diagonal
11,NaN,NaN,coffee_a,12,coffee_a,NaN,NaN,NaN
13,NaN,NaN,coffee_b,2,coffee_a,price_dollars,$12,first
15,NaN,NaN,coffee_b,4,coffee_a,price_cents,.99,alternative
17,NaN,NaN,coffee_b,6,coffee_a,weight_oz,13.0 oz,alternative
19,NaN,NaN,coffee_b,8,coffee_b,price_dollars,$13,diagonal


In [22]:
gpt = df[(df['model_name'].str.contains('gpt'))&(df['kind']=='tool')]
# gemini_lite.columns
gpt[['model_name', 'budget_max','choice','step_index','option_id', 'attribute_id', 'value', 'transition']]


,model_name,budget_max,choice,step_index,option_id,attribute_id,value,transition
115,gpt-5.4-mini,5.0,coffee_b,2,coffee_a,price_dollars,$12,first
117,gpt-5.4-mini,5.0,coffee_b,4,coffee_a,price_cents,.99,alternative
119,gpt-5.4-mini,5.0,coffee_b,6,coffee_a,weight_oz,13.0 oz,alternative
121,gpt-5.4-mini,5.0,coffee_b,8,coffee_b,price_dollars,$13,diagonal
123,gpt-5.4-mini,5.0,coffee_b,10,coffee_b,weight_oz,13.9 oz,alternative
125,gpt-5.4-mini,5.0,coffee_b,12,coffee_b,NaN,NaN,NaN
127,gpt-5.4,5.0,coffee_b,2,coffee_a,price_dollars,$12,first
129,gpt-5.4,5.0,coffee_b,4,coffee_a,price_cents,.99,alternative
131,gpt-5.4,5.0,coffee_b,6,coffee_a,weight_oz,13.0 oz,alternative
133,gpt-5.4,5.0,coffee_b,8,coffee_b,price_dollars,$13,diagonal


In [34]:
df.columns

Index(['session_name', 'experiment_name', 'provider', 'model_name',
       'forced_choice', 'choice', 'cumulative_cost_usd',
       'budget_remaining_usd', 'cumulative_cost_tools',
       'budget_remaining_tools', 'option_count', 'seed', 'started_at',
       'finished_at', 'step_index', 'kind', 'content', 'reasoning',
       'input_tokens', 'output_tokens', 'input_cost', 'output_cost',
       'tool_cost', 'tool_calls', 'finish_reason', 'tool_name', 'tool_call_id',
       'option_id', 'attribute_id', 'value', 'turn_cost_tools', 'is_revisit',
       'transition', 'option_label', 'budget_type', 'budget_max',
       'budget_remaining_tokens', 'cumulative_cost_tokens', 'turn_cost_tokens',
       'meta', 'meta_gpt-5.4-mini-2026-03-17', 'meta_none',
       'meta_gpt-5.4-2026-03-05', 'meta_gpt-5.4-nano-2026-03-17',
       'meta_claude-sonnet-4-6', 'meta_thinking', 'meta_effort',
       'meta_model_version'],
      dtype='str')

In [35]:
df

,session_name,experiment_name,provider,model_name,forced_choice,choice,cumulative_cost_usd,budget_remaining_usd,cumulative_cost_tools,budget_remaining_tools,...,turn_cost_tokens,meta,meta_gpt-5.4-mini-2026-03-17,meta_none,meta_gpt-5.4-2026-03-05,meta_gpt-5.4-nano-2026-03-17,meta_claude-sonnet-4-6,meta_thinking,meta_effort,meta_model_version
0,20260409T201335248464Z,left-digit,google,gemini-3-flash-preview,False,coffee_a,0.000309,-0.000309,1,4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,20260409T201335248464Z,left-digit,google,gemini-3-flash-preview,False,coffee_a,0.000309,-0.000309,1,4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,20260409T201335248464Z,left-digit,google,gemini-3-flash-preview,False,coffee_a,0.001107,-0.001107,2,3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,20260409T201335248464Z,left-digit,google,gemini-3-flash-preview,False,coffee_a,0.001107,-0.001107,2,3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,20260409T201335248464Z,left-digit,google,gemini-3-flash-preview,False,coffee_a,0.002083,-0.002083,3,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
225,20260410T163414801115Z,left-digit,llamacpp,gemma-4-26B-A4B-it-UD-Q4_K_XL.gguf,False,coffee_b,0.000000,0.000000,4,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,gemma-4-26B-A4B-it-UD-Q4_K_XL.gguf
226,20260410T163414801115Z,left-digit,llamacpp,gemma-4-26B-A4B-it-UD-Q4_K_XL.gguf,False,coffee_b,0.000000,0.000000,5,0,...,NaN,{'model_version': 'gemma-4-26B-A4B-it-UD-Q4_K_...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,gemma-4-26B-A4B-it-UD-Q4_K_XL.gguf
227,20260410T163414801115Z,left-digit,llamacpp,gemma-4-26B-A4B-it-UD-Q4_K_XL.gguf,False,coffee_b,0.000000,0.000000,5,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,gemma-4-26B-A4B-it-UD-Q4_K_XL.gguf
228,20260410T163414801115Z,left-digit,llamacpp,gemma-4-26B-A4B-it-UD-Q4_K_XL.gguf,False,coffee_b,0.000000,0.000000,5,0,...,NaN,{'model_version': 'gemma-4-26B-A4B-it-UD-Q4_K_...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,gemma-4-26B-A4B-it-UD-Q4_K_XL.gguf


In [32]:
anthropic = df[(df['provider'].str.contains('anthropic'))&(df['kind']=='tool')]
# gemini_lite.columns
anthropic[['model_name', 'meta_thinking', 'meta_effort', 'budget_max','choice','step_index','option_id', 'attribute_id', 'value', 'transition']]


,model_name,meta_thinking,meta_effort,budget_max,choice,step_index,option_id,attribute_id,value,transition
175,claude-sonnet-4-6,NaN,NaN,5.0,coffee_b,2,coffee_a,price_dollars,$12,first
177,claude-sonnet-4-6,NaN,NaN,5.0,coffee_b,4,coffee_a,price_cents,.99,alternative
179,claude-sonnet-4-6,NaN,NaN,5.0,coffee_b,6,coffee_a,weight_oz,13.0 oz,alternative
181,claude-sonnet-4-6,NaN,NaN,5.0,coffee_b,8,coffee_b,price_dollars,$13,diagonal
183,claude-sonnet-4-6,NaN,NaN,5.0,coffee_b,10,coffee_b,weight_oz,13.9 oz,alternative
185,claude-sonnet-4-6,NaN,NaN,5.0,coffee_b,12,coffee_b,NaN,NaN,NaN
187,claude-sonnet-4-6,NaN,NaN,5.0,coffee_a,2,coffee_a,price_dollars,$12,first
189,claude-sonnet-4-6,NaN,NaN,5.0,coffee_a,4,coffee_b,price_dollars,$13,attribute
191,claude-sonnet-4-6,NaN,NaN,5.0,coffee_a,6,coffee_a,weight_oz,13.0 oz,diagonal
193,claude-sonnet-4-6,NaN,NaN,5.0,coffee_a,8,coffee_b,weight_oz,13.9 oz,attribute


In [3]:
scores = {opt['id']:opt['base_score'] for opt in config['options']}
scores

{'coffee_a': 1.00077, 'coffee_b': 1.06923}

In [4]:
# df['budget_type']

In [5]:
session_choices = df.groupby(['model_name','session_name'])['choice'].last().reset_index()
session_choices['score'] = session_choices['choice'].map(scores)
session_choices

,model_name,session_name,choice,score
0,gemini-3-flash-preview,20260409T201335248464Z,coffee_a,1.00077
1,gemini-3-flash-preview,20260409T202110975586Z,coffee_b,1.06923
2,gemini-3-flash-preview,20260409T213551363719Z,coffee_a,1.00077
3,gemini-3-flash-preview,20260409T224320378210Z,coffee_a,1.00077
4,gemini-3.1-flash-lite-preview,20260409T205849239956Z,coffee_a,1.00077
5,gemini-3.1-flash-lite-preview,20260409T205948438333Z,coffee_a,1.00077
6,gemini-3.1-flash-lite-preview,20260409T212306149951Z,coffee_a,1.00077
7,gemini-3.1-flash-lite-preview,20260409T212652870379Z,coffee_a,1.00077
8,gemini-3.1-flash-lite-preview,20260409T213049600406Z,coffee_b,1.06923
9,gemini-3.1-flash-lite-preview,20260409T213418124240Z,coffee_a,1.00077


In [6]:
(df[(df['choice']=='coffee_b')&(df['tool_name']=='inspect_cell')])[['step_index','option_id', 'attribute_id', 'value']]

,step_index,option_id,attribute_id,value
13,2,coffee_a,price_dollars,$12
15,4,coffee_a,price_cents,.99
17,6,coffee_a,weight_oz,13.0 oz
19,8,coffee_b,price_dollars,$13
21,10,coffee_b,weight_oz,13.9 oz
76,2,coffee_a,price_dollars,$12
78,4,coffee_a,price_cents,.99
80,6,coffee_a,weight_oz,13.0 oz
82,8,coffee_b,price_dollars,$13
84,10,coffee_b,price_cents,.00


In [84]:
13/12.99 # value

1.0007698229407236

In [85]:
print('worst case',13.99/13.99) # worst case
print('best case', 13.99/13.00) # best case

worst case 1.0
best case 1.0761538461538462


KeyError: "['budget_max'] not in index"